# Generación de hologramas de eritrocitos a partir de mapas de fase

Este notebook entrena una GAN condicional que recibe el mapa de fase de un eritrocito y genera su holograma. La calidad del holograma se mide reconstruyendo su fase con la transformada de Fourier y comparándola con la fase de entrada.

Se ejecuta de arriba hacia abajo. Las secciones 1 y 2 preparan funciones y datos, la sección 3 calcula la referencia, la sección 4 entrena y la sección 5 evalúa.

# 0. Importaciones y rutas

In [ ]:
import json
import os
import random
import re
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.fft as tfft
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.utils import save_image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)

In [ ]:
RUTA_DATASET = Path("data")                 # debe contener las carpetas Holograms y Phase
RUTA_SALIDA = Path("resultados")            # muestras, checkpoints, curvas y tablas, una subcarpeta por experimento
RUTA_SPLIT = Path("splits/split.csv")       # partición train, val y test por holograma original

RUTA_SALIDA.mkdir(parents=True, exist_ok=True)
print("Dataset:", RUTA_DATASET.resolve())
print("Salida :", RUTA_SALIDA.resolve())

# 1. Funciones

## 1.1 Datos

Lectura de imágenes, emparejamiento de cada holograma con su fase, partición y cargadores de datos. Cuando un par tiene la marca de volteo, la fase y el holograma se voltean horizontalmente al cargarlos.

In [ ]:
PATRON_NOMBRE = re.compile(
    r"Image__(\d{4}-\d{2}-\d{2})__(\d{2}-\d{2}-\d{2})_(\d+)-(\d+)(?:\.png)?(?:-([A-Za-z]))?(?:\.png)+"
)


def leer_png(ruta):
    """Imagen 2D en [0, 1]. Soporta PNG de 8 y de 16 bits."""
    img = Image.open(ruta)
    if img.mode in ("I;16", "I;16B", "I"):
        return np.array(img, dtype=np.float32) / 65535.0
    return np.array(img.convert("L"), dtype=np.float32) / 255.0


def indexar_carpeta(ruta_base, carpetas=None):
    """Tabla con una fila por imagen. Si el nombre no trae letra de versión, la versión es orig."""
    ruta_base = Path(ruta_base)
    if carpetas is None:
        carpetas = sorted(p.name for p in ruta_base.iterdir() if p.is_dir())
    filas, otros = [], []
    for carpeta in carpetas:
        for ruta in sorted((ruta_base / carpeta).glob("*.png")):
            m = PATRON_NOMBRE.fullmatch(ruta.name)
            if m is None:
                otros.append(ruta.name)
                continue
            fecha, hora, a, b, letra = m.groups()
            filas.append(dict(carpeta=carpeta, archivo=ruta.name, ruta=ruta,
                              original=f"{fecha}__{hora}", a=int(a), b=int(b),
                              variante=letra or "orig"))
    if otros:
        print(f"{ruta_base.name}: {len(otros)} archivos con otro formato de nombre, primeros 5: {otros[:5]}")
    return pd.DataFrame(filas)


def emparejar(df_holo, df_fase):
    """Une holograma y fase por original, recorte y versión, y reporta lo que no coincide."""
    if df_holo.empty or df_fase.empty:
        raise ValueError("No se reconoció ningún nombre de archivo en Holograms o en Phase.")
    claves = ["original", "a", "b", "variante"]
    union = df_holo.merge(df_fase, on=claves, how="outer", suffixes=("_holo", "_fase"), indicator=True)
    print("Pares completos:", int((union["_merge"] == "both").sum()))
    print("Solo holograma:", int((union["_merge"] == "left_only").sum()),
          "| Solo fase:", int((union["_merge"] == "right_only").sum()))
    pares = union[union["_merge"] == "both"].drop(columns="_merge").copy()
    pares = pares.rename(columns={"carpeta_holo": "carpeta", "archivo_holo": "archivo"})
    return pares.reset_index(drop=True)


def parches_en_ambas_carpetas(df):
    """Recortes con versiones en más de una carpeta, y el total de recortes."""
    n = df.groupby(["original", "a", "b"])["carpeta"].nunique()
    return n[n > 1].index, len(n)


def partir_por_original(df, proporciones=(0.70, 0.15, 0.15), semilla=42, ruta_csv=None):
    """Reparte los hologramas originales entre train, val y test.
    Todo lo que viene de un mismo original queda en el mismo conjunto.
    Si ruta_csv existe se carga y se usa tal cual. Si no, se crea y se guarda."""
    if ruta_csv is not None and Path(ruta_csv).exists():
        tabla = pd.read_csv(ruta_csv)
        print("Partición cargada de", ruta_csv)
    else:
        originales = np.array(sorted(df["original"].unique()))
        np.random.default_rng(semilla).shuffle(originales)
        n = len(originales)
        n_val, n_test = round(proporciones[1] * n), round(proporciones[2] * n)
        n_train = n - n_val - n_test
        tabla = pd.DataFrame({"original": originales,
                              "particion": ["train"] * n_train + ["val"] * n_val + ["test"] * n_test})
        if ruta_csv is not None:
            Path(ruta_csv).parent.mkdir(parents=True, exist_ok=True)
            tabla.sort_values("original").to_csv(ruta_csv, index=False)
            print("Partición creada y guardada en", ruta_csv)
    df = df.merge(tabla, on="original", how="left")
    assert df["particion"].notna().all(), "Hay hologramas originales sin partición asignada"
    return df


class ParesDataset(Dataset):
    """Entrega fase y holograma, ambos de forma (1, 256, 256) y en [-1, 1].
    Si voltear[i] es verdadero, los dos se voltean horizontalmente."""
    def __init__(self, rutas_fase, rutas_holo, voltear):
        assert len(rutas_fase) == len(rutas_holo) == len(voltear)
        self.rutas_fase = list(rutas_fase)
        self.rutas_holo = list(rutas_holo)
        self.voltear = list(voltear)

    def __len__(self):
        return len(self.rutas_fase)

    def __getitem__(self, i):
        fase = leer_png(self.rutas_fase[i]) * 2 - 1
        holo = leer_png(self.rutas_holo[i]) * 2 - 1
        if self.voltear[i]:
            fase, holo = fase[:, ::-1], holo[:, ::-1]
        fase, holo = np.ascontiguousarray(fase), np.ascontiguousarray(holo)
        return torch.from_numpy(fase).unsqueeze(0), torch.from_numpy(holo).unsqueeze(0)


def crear_loader(df, cfg, entrenamiento=True):
    """DataLoader de pares para las filas de df."""
    ds = ParesDataset(df["ruta_fase"].tolist(), df["ruta_holo"].tolist(), df["voltear"].tolist())
    return DataLoader(ds, batch_size=cfg["batch"], shuffle=entrenamiento, num_workers=0, drop_last=entrenamiento)


def cargar_pares(df, n=None):
    """n pares equiespaciados de df como tensores (n, 1, 256, 256). Con n=None usa todos."""
    n = len(df) if n is None else min(n, len(df))
    idx = np.linspace(0, len(df) - 1, n).astype(int)
    sub = df.iloc[idx]
    ds = ParesDataset(sub["ruta_fase"].tolist(), sub["ruta_holo"].tolist(), sub["voltear"].tolist())
    fases, holos = zip(*[ds[i] for i in range(len(ds))])
    return torch.stack(fases), torch.stack(holos)


def mostrar_pares(df, indices):
    """Fase y holograma lado a lado, para comprobar a ojo que corresponden."""
    fig, ejes = plt.subplots(2, len(indices), figsize=(4 * len(indices), 8))
    ejes = np.asarray(ejes).reshape(2, -1)
    for j, i in enumerate(indices):
        f = df.iloc[i]
        ejes[0, j].imshow(leer_png(f.ruta_fase), cmap="gray")
        ejes[0, j].set_title("Fase\n" + f.archivo_fase, fontsize=7)
        ejes[1, j].imshow(leer_png(f.ruta_holo), cmap="gray")
        ejes[1, j].set_title("Holograma\n" + f.archivo, fontsize=7)
        ejes[0, j].axis("off"); ejes[1, j].axis("off")
    plt.tight_layout()
    plt.show()

## 1.2 Análisis de Fourier

Un holograma off-axis tiene en su transformada de Fourier un pico central y dos picos laterales. El pico lateral contiene la información de la fase.

Las funciones de esta sección buscan el pico lateral, reconstruyen la fase filtrando alrededor de ese pico y la comparan con la fase de referencia.

Los mapas de fase del dataset no salieron de una FFT simple, así que no se comparan píxel a píxel. Se ajusta la fase reconstruida como a por la fase de referencia más un desfase y una inclinación, por mínimos cuadrados, y se mide lo que ese ajuste no explica. El r2 es la fracción de la variación de la fase reconstruida que explica la referencia, y vale 1 si coinciden. El nrmse es el error del ajuste dividido por la variación de la referencia escalada, y vale 0 si coinciden.

El resultado con el holograma real es el piso, lo mejor que se puede lograr con este método. El holograma generado se compara contra ese piso.

In [ ]:
RADIO_CENTRO = 15   # píxeles que se ignoran alrededor del pico central al buscar el pico lateral


def espectro_log(img):
    """Magnitud del espectro en escala log, con la frecuencia cero en el centro."""
    return np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(img))))


def ver_fourier(imagen, titulo=None):
    """Muestra una imagen y su transformada de Fourier."""
    imagen = np.asarray(imagen, dtype=np.float32)
    fig, ejes = plt.subplots(1, 2, figsize=(10, 5))
    ejes[0].imshow(imagen, cmap="gray")
    ejes[0].set_title(titulo or "Imagen")
    ejes[0].axis("off")
    ejes[1].imshow(espectro_log(imagen), cmap="gray")
    ejes[1].set_title("Transformada de Fourier")
    ejes[1].axis("off")
    plt.tight_layout()
    plt.show()


def metricas_picos(img, radio_centro=None):
    """Pico lateral más fuerte. Devuelve distancia al centro, ángulo entre 0 y 180 y prominencia."""
    radio_centro = RADIO_CENTRO if radio_centro is None else radio_centro
    img = img - img.mean()
    mag = np.abs(np.fft.fftshift(np.fft.fft2(img)))
    h, w = mag.shape
    cy, cx = h // 2, w // 2
    yy, xx = np.ogrid[:h, :w]
    dist = np.hypot(yy - cy, xx - cx)

    fuera = mag.copy()
    fuera[dist < radio_centro] = 0
    iy, ix = np.unravel_index(np.argmax(fuera), fuera.shape)
    fondo = np.median(mag[dist >= radio_centro])
    return {
        "dist": float(dist[iy, ix]),
        "angulo": float(np.degrees(np.arctan2(cy - iy, ix - cx)) % 180),
        "prominencia": float(fuera[iy, ix] / fondo),
        "_pos": (int(iy), int(ix)),
    }


def tabla_picos(imagenes):
    """DataFrame con las métricas de picos de cada imagen."""
    return pd.DataFrame([metricas_picos(im) for im in imagenes]).drop(columns="_pos")


def verificar_radio(img, radio_centro=None):
    """Dibuja la zona ignorada y el pico detectado."""
    radio_centro = RADIO_CENTRO if radio_centro is None else radio_centro
    m = metricas_picos(img, radio_centro)
    h, w = img.shape
    fig, eje = plt.subplots(figsize=(6, 6))
    eje.imshow(espectro_log(img), cmap="gray")
    eje.add_patch(plt.Circle((w // 2, h // 2), radio_centro, fill=False, color="cyan", lw=1.5))
    eje.plot(m["_pos"][1], m["_pos"][0], "r+", ms=18, mew=2)
    eje.set_title(f"dist={m['dist']:.1f}  prominencia={m['prominencia']:.0f}")
    eje.axis("off")
    plt.show()


def reconstruir_fase(holo, radio_filtro=None, radio_centro=None):
    """Fase envuelta en radianes de un holograma off-axis 2D, filtrando el pico lateral.
    radio_filtro es el radio del filtro en píxeles. Por defecto, un tercio de la distancia al pico."""
    holo = np.asarray(holo, dtype=np.float64)
    h, w = holo.shape
    m = metricas_picos(holo, radio_centro)
    iy, ix = m["_pos"]
    cy, cx = h // 2, w // 2
    radio = radio_filtro if radio_filtro is not None else m["dist"] / 3
    F = np.fft.fftshift(np.fft.fft2(holo - holo.mean()))
    yy, xx = np.ogrid[:h, :w]
    F = np.where(np.hypot(yy - iy, xx - ix) <= radio, F, 0)
    F = np.roll(F, (cy - iy, cx - ix), axis=(0, 1))          # lleva el pico al centro
    campo = np.fft.ifft2(np.fft.ifftshift(F))

    # Queda una inclinación de fase residual porque el pico no cae justo en un píxel.
    # Se estima con el gradiente medio del campo complejo y se quita.
    gx = np.angle(np.sum(campo[:, 1:] * np.conj(campo[:, :-1])))
    gy = np.angle(np.sum(campo[1:, :] * np.conj(campo[:-1, :])))
    yy2, xx2 = np.mgrid[:h, :w]
    campo = campo * np.exp(-1j * (gx * xx2 + gy * yy2))
    return np.angle(campo)


def comparar_fase(fase_rec, fase_ref):
    """Ajusta fase_rec ~ a*fase_ref + b + c*x + d*y y mide lo que el ajuste no explica."""
    h, w = fase_ref.shape
    yy, xx = np.mgrid[:h, :w]
    A = np.column_stack([fase_ref.ravel(), np.ones(h * w), xx.ravel() / w, yy.ravel() / h])
    coef, *_ = np.linalg.lstsq(A, fase_rec.ravel(), rcond=None)
    resid = fase_rec.ravel() - A @ coef
    rec_sin_molestias = fase_rec.ravel() - A[:, 1:] @ coef[1:]    # sin desfase ni inclinación
    var_rec = rec_sin_molestias.var()
    escala_ref = np.abs(coef[0]) * fase_ref.ravel().std()
    return {
        "r2": float(1 - resid.var() / var_rec) if var_rec > 0 else np.nan,
        "nrmse": float(np.sqrt((resid ** 2).mean()) / escala_ref) if escala_ref > 0 else np.nan,
        "escala": float(coef[0]),
    }


def verificar_reconstruccion(holo, fase_ref):
    """Holograma, fase reconstruida con la FFT y fase de referencia."""
    rec = reconstruir_fase(holo)
    met = comparar_fase(rec, fase_ref)
    fig, ejes = plt.subplots(1, 3, figsize=(15, 5))
    ejes[0].imshow(holo, cmap="gray"); ejes[0].set_title("Holograma")
    ejes[1].imshow(rec, cmap="gray"); ejes[1].set_title("Fase reconstruida con FFT")
    ejes[2].imshow(fase_ref, cmap="gray"); ejes[2].set_title(f"Fase de referencia\nr2={met['r2']:.2f}  nrmse={met['nrmse']:.2f}")
    for e in ejes:
        e.axis("off")
    plt.tight_layout()
    plt.show()
    return met

## 1.3 Modelos pix2pix

Se usa pix2pix, de Isola, Zhu, Zhou y Efros, con la arquitectura de la implementación de referencia junyanz/pytorch-CycleGAN-and-pix2pix.

El generador es una U-Net de 8 niveles. Reduce la imagen de 256 por 256 a 1 por 1 y la vuelve a ampliar. Las conexiones de salto llevan los detalles finos del codificador al decodificador, y esto importa porque las franjas del holograma son detalle fino.

El discriminador es un PatchGAN de 70 por 70. Da una cuadrícula de 30 por 30 veredictos, uno por parche. Recibe la fase y el holograma juntos, así que juzga si el holograma es coherente con esa fase.

In [ ]:
class UnetSkip(nn.Module):
    """Un nivel de la U-Net. Baja, llama al nivel interior, sube y concatena con la entrada."""
    def __init__(self, ext_nc, int_nc, sub=None, input_nc=None, outermost=False, innermost=False, dropout=False):
        super().__init__()
        self.outermost = outermost
        input_nc = input_nc or ext_nc
        down = nn.Conv2d(input_nc, int_nc, 4, 2, 1, bias=False)

        if outermost:
            up = nn.ConvTranspose2d(int_nc * 2, ext_nc, 4, 2, 1)
            modelo = [down, sub, nn.ReLU(True), up, nn.Tanh()]
        elif innermost:
            up = nn.ConvTranspose2d(int_nc, ext_nc, 4, 2, 1, bias=False)
            modelo = [nn.LeakyReLU(0.2, True), down, nn.ReLU(True), up, nn.BatchNorm2d(ext_nc)]
        else:
            up = nn.ConvTranspose2d(int_nc * 2, ext_nc, 4, 2, 1, bias=False)
            modelo = [nn.LeakyReLU(0.2, True), down, nn.BatchNorm2d(int_nc), sub,
                      nn.ReLU(True), up, nn.BatchNorm2d(ext_nc)]
            if dropout:
                modelo.append(nn.Dropout(0.5))
        self.modelo = nn.Sequential(*modelo)

    def forward(self, x):
        y = self.modelo(x)
        return y if self.outermost else torch.cat([x, y], 1)


class GeneradorUNet(nn.Module):
    """U-Net de 8 niveles para imágenes de 256 por 256."""
    def __init__(self, in_nc=1, out_nc=1, ngf=64):
        super().__init__()
        bloque = UnetSkip(ngf * 8, ngf * 8, innermost=True)
        for _ in range(3):
            bloque = UnetSkip(ngf * 8, ngf * 8, sub=bloque, dropout=True)
        bloque = UnetSkip(ngf * 4, ngf * 8, sub=bloque)
        bloque = UnetSkip(ngf * 2, ngf * 4, sub=bloque)
        bloque = UnetSkip(ngf, ngf * 2, sub=bloque)
        self.modelo = UnetSkip(out_nc, ngf, sub=bloque, input_nc=in_nc, outermost=True)

    def forward(self, x):
        return self.modelo(x)


class DiscriminadorPatch(nn.Module):
    """PatchGAN 70x70. Entrada: fase y holograma concatenados. Salida: cuadrícula de 30x30."""
    def __init__(self, in_nc=2, ndf=64):
        super().__init__()

        def bloque(entrada, salida, stride):
            return [nn.Conv2d(entrada, salida, 4, stride, 1, bias=False),
                    nn.BatchNorm2d(salida), nn.LeakyReLU(0.2, True)]

        self.modelo = nn.Sequential(
            nn.Conv2d(in_nc, ndf, 4, 2, 1), nn.LeakyReLU(0.2, True),
            *bloque(ndf, ndf * 2, 2),
            *bloque(ndf * 2, ndf * 4, 2),
            *bloque(ndf * 4, ndf * 8, 1),
            nn.Conv2d(ndf * 8, 1, 4, 1, 1),
        )

    def forward(self, fase, holo):
        return self.modelo(torch.cat([fase, holo], dim=1))


def inicializar_pesos(m):
    """Inicialización del original, normal de media 0 y desviación 0.02."""
    nombre = m.__class__.__name__
    if "Conv" in nombre:
        nn.init.normal_(m.weight, 0.0, 0.02)
    elif "BatchNorm" in nombre:
        nn.init.normal_(m.weight, 1.0, 0.02)
        nn.init.zeros_(m.bias)

## 1.4 Pérdida de consistencia de fase

Se quiere que del holograma generado se pueda recuperar la fase de entrada. La pérdida repite en PyTorch lo que hace la reconstrucción de la sección 1.2, pero con un pico lateral fijo para poder derivar. Filtra alrededor del pico, lo lleva al centro, quita la inclinación y toma el ángulo.

Después compara esa fase con la de entrada en forma compleja, con la expresión 1 menos el módulo de la media de exp(i por la diferencia). El espectro tiene dos picos laterales que dan fases de signo opuesto, así que se prueban los dos signos y se queda el mejor. Un desplazamiento global de las franjas solo suma una constante a la fase y no se castiga.

La posición del pico, el radio del filtro y la escala de la fase se estiman en la sección 3.

In [ ]:
# POS_PICO, RADIO_F y A_ABS se calculan en la sección 3.3 antes de entrenar.

def _rejilla(H, W, dev, dtype):
    yy, xx = torch.meshgrid(torch.arange(H, device=dev, dtype=dtype),
                            torch.arange(W, device=dev, dtype=dtype), indexing="ij")
    return yy, xx


def _quitar_inclinacion(c):
    """Quita la inclinación de fase residual de un campo complejo de forma (B, H, W)."""
    gx = torch.angle((c[:, :, 1:] * c[:, :, :-1].conj()).sum(dim=(-2, -1)))
    gy = torch.angle((c[:, 1:, :] * c[:, :-1, :].conj()).sum(dim=(-2, -1)))
    yy, xx = _rejilla(c.shape[-2], c.shape[-1], c.device, gx.dtype)
    return c * torch.exp(-1j * (gx[:, None, None] * xx + gy[:, None, None] * yy))


def reconstruir_fase_torch(holo, pos=None, radio=None):
    """Holograma (B, 1, H, W) a fase envuelta (B, H, W), con el pico lateral fijo."""
    pos = POS_PICO if pos is None else pos
    radio = RADIO_F if radio is None else radio
    h = holo.squeeze(1)
    B, H, W = h.shape
    cy, cx = H // 2, W // 2
    iy, ix = pos
    F = tfft.fftshift(tfft.fft2(h - h.mean(dim=(-2, -1), keepdim=True)), dim=(-2, -1))
    yy, xx = _rejilla(H, W, h.device, h.dtype)
    F = F * (((yy - iy) ** 2 + (xx - ix) ** 2) <= radio ** 2)
    F = torch.roll(F, shifts=(cy - iy, cx - ix), dims=(-2, -1))
    campo = tfft.ifft2(tfft.ifftshift(F, dim=(-2, -1)))
    return torch.angle(_quitar_inclinacion(campo))


def perdida_fase_torch(rec, ref, a_abs=None):
    """rec es la fase reconstruida (B, H, W) y ref la fase de entrada (B, 1, H, W). Devuelve (B,)."""
    a_abs = A_ABS if a_abs is None else a_abs
    ref = ref.squeeze(1).to(rec.dtype)
    Ls = []
    for s in (1.0, -1.0):
        d = _quitar_inclinacion(torch.exp(1j * (rec - s * a_abs * ref)))
        Ls.append(1 - d.mean(dim=(-2, -1)).abs())
    return torch.stack(Ls).min(dim=0).values

## 1.5 Entrenamiento

En cada lote el discriminador ve el par de fase y holograma real con etiqueta 1, y el par de fase y holograma generado con etiqueta 0. El generador busca engañar al discriminador, parecerse al holograma real y que su fase se pueda recuperar. Su pérdida es la adversaria más lambda_l1 por la L1 más lambda_fase por la pérdida de fase.

Al final de cada época se miden sobre validación la L1, la prominencia del pico lateral, el error de fase y la pérdida de fase. Se guarda el generador cada cierto número de épocas y un checkpoint completo para retomar el entrenamiento. Además, al final de cada época se guarda en la carpeta muestras una imagen con cuatro pares fijos de validación: fases, hologramas generados y reales.

In [ ]:
CLAVES_HIST = ["loss_D", "loss_G_gan", "loss_L1", "D_real", "D_falsa", "loss_fase",
               "val_L1", "val_prominencia", "val_nrmse_fase", "val_L_fase", "tiempo"]
crit_gan = nn.BCEWithLogitsLoss()
crit_l1 = nn.L1Loss()


@torch.no_grad()
def predecir(G, fases, batch=16):
    """Hologramas generados para un tensor de fases (n, 1, 256, 256), con G en modo evaluación."""
    estado = G.training
    G.eval()
    salida = [G(fases[i:i + batch].to(device)).cpu() for i in range(0, len(fases), batch)]
    G.train(estado)
    return torch.cat(salida)


def metricas_val(exp):
    """L1, prominencia del pico, error de fase y pérdida de fase sobre el subconjunto fijo de validación."""
    fases, holos = exp["val_fase"], exp["val_holo"]
    pred = predecir(exp["G"], fases)
    val_l1 = float((pred - holos).abs().mean())
    pred_np = pred.squeeze(1).numpy()
    prom = float(tabla_picos(pred_np[:64])["prominencia"].mean())
    nrmse = [comparar_fase(reconstruir_fase(pred_np[i]), fases[i, 0].numpy())["nrmse"]
             for i in range(min(32, len(pred_np)))]
    with torch.no_grad():
        l_fase = perdida_fase_torch(reconstruir_fase_torch(pred.to(device)), fases.to(device)).median().item()
    return val_l1, prom, float(np.nanmedian(nrmse)), l_fase


def crear_experimento(cfg, df_val):
    """Modelos, optimizadores, historial y subconjunto fijo de validación."""
    torch.manual_seed(cfg["semilla"])
    np.random.seed(cfg["semilla"])
    random.seed(cfg["semilla"])

    carpeta = RUTA_SALIDA / cfg["nombre"]
    (carpeta / "muestras").mkdir(parents=True, exist_ok=True)
    (carpeta / "checkpoints").mkdir(parents=True, exist_ok=True)
    (carpeta / "config.json").write_text(json.dumps(cfg, indent=2))

    G = GeneradorUNet(1, 1, cfg["ngf"]).to(device)
    D = DiscriminadorPatch(2, cfg["ndf"]).to(device)
    G.apply(inicializar_pesos)
    D.apply(inicializar_pesos)

    val_fase, val_holo = cargar_pares(df_val, cfg["n_val_epoca"])
    return dict(
        cfg=cfg, carpeta=carpeta, G=G, D=D,
        opt_G=optim.Adam(G.parameters(), lr=cfg["lr_G"], betas=(cfg["beta1"], 0.999)),
        opt_D=optim.Adam(D.parameters(), lr=cfg["lr_D"], betas=(cfg["beta1"], 0.999)),
        val_fase=val_fase, val_holo=val_holo,
        hist={k: [] for k in CLAVES_HIST},
    )


def entrenar(exp, loader, epocas, max_lotes=None):
    """Entrena epocas más, continuando desde la última época registrada.
    max_lotes limita los lotes por época y solo sirve para la prueba rápida."""
    cfg, G, D = exp["cfg"], exp["G"], exp["D"]
    opt_G, opt_D, hist, carpeta = exp["opt_G"], exp["opt_D"], exp["hist"], exp["carpeta"]
    inicio = len(hist["loss_D"])
    lam_l1, lam_fase = cfg["lambda_l1"], cfg["lambda_fase"]

    for epoca in range(inicio, inicio + epocas):
        t0 = time.time()
        G.train(); D.train()
        acum, n = np.zeros(6), 0

        for i, (fase, holo) in enumerate(loader):
            if max_lotes is not None and i >= max_lotes:
                break
            fase, holo = fase.to(device), holo.to(device)
            falsa = G(fase)

            # Discriminador
            opt_D.zero_grad()
            pred_real = D(fase, holo)
            pred_falsa = D(fase, falsa.detach())
            loss_D = 0.5 * (crit_gan(pred_real, torch.ones_like(pred_real)) +
                            crit_gan(pred_falsa, torch.zeros_like(pred_falsa)))
            loss_D.backward()
            opt_D.step()

            # Generador
            opt_G.zero_grad()
            pred = D(fase, falsa)
            loss_gan = crit_gan(pred, torch.ones_like(pred))
            loss_l1 = crit_l1(falsa, holo)
            loss_fase = perdida_fase_torch(reconstruir_fase_torch(falsa), fase).mean()
            (loss_gan + lam_l1 * loss_l1 + lam_fase * loss_fase).backward()
            opt_G.step()

            acum += [loss_D.item(), loss_gan.item(), loss_l1.item(),
                     torch.sigmoid(pred_real).mean().item(),
                     torch.sigmoid(pred_falsa).mean().item(), loss_fase.item()]
            n += 1

        prom = acum / n
        for clave, valor in zip(["loss_D", "loss_G_gan", "loss_L1", "D_real", "D_falsa", "loss_fase"], prom):
            hist[clave].append(float(valor))

        val_l1, val_prom, val_nrmse, val_lf = metricas_val(exp)
        hist["val_L1"].append(val_l1)
        hist["val_prominencia"].append(val_prom)
        hist["val_nrmse_fase"].append(val_nrmse)
        hist["val_L_fase"].append(val_lf)
        hist["tiempo"].append(float(time.time() - t0))

        # Muestras de 4 pares fijos (fases, generados, reales), una imagen por época
        fases4, reales4 = exp["val_fase"][:4], exp["val_holo"][:4]
        gen4 = predecir(G, fases4)
        save_image(torch.cat([fases4, gen4, reales4]), carpeta / "muestras" / f"epoca_{epoca + 1:03d}.png",
                   nrow=4, normalize=True, value_range=(-1, 1))

        if (epoca + 1) % cfg["guardar_cada"] == 0:
            torch.save(G.state_dict(), carpeta / "checkpoints" / f"G_epoca_{epoca + 1:03d}.pt")
        tmp = carpeta / "checkpoints" / "ultimo_tmp.pt"
        torch.save({"epoca": epoca + 1, "G": G.state_dict(), "D": D.state_dict(),
                    "opt_G": opt_G.state_dict(), "opt_D": opt_D.state_dict(),
                    "hist": hist, "cfg": cfg}, tmp)
        os.replace(tmp, carpeta / "checkpoints" / "ultimo.pt")
        pd.DataFrame(hist).to_csv(carpeta / "historial.csv", index_label="epoca")

        print(f"Época {epoca + 1} | D: {prom[0]:.3f} | G_gan: {prom[1]:.3f} | L1 train: {prom[2]:.4f} | "
              f"L1 val: {val_l1:.4f} | L_fase train: {prom[5]:.3f} val: {val_lf:.3f} | "
              f"D(real): {prom[3]:.2f} D(falsa): {prom[4]:.2f} | nrmse fase: {val_nrmse:.2f} | "
              f"{hist['tiempo'][-1]:.0f} s")


def cargar_checkpoint(exp, ruta=None):
    """Retoma un experimento desde ultimo.pt."""
    ruta = ruta or exp["carpeta"] / "checkpoints" / "ultimo.pt"
    ck = torch.load(ruta, map_location=device, weights_only=False)
    exp["G"].load_state_dict(ck["G"])
    exp["D"].load_state_dict(ck["D"])
    exp["opt_G"].load_state_dict(ck["opt_G"])
    exp["opt_D"].load_state_dict(ck["opt_D"])
    exp["hist"] = ck["hist"]
    print(f"Retomado desde la época {ck['epoca']}")


def cargar_generador(ruta, ngf=64):
    """Generador con los pesos guardados en un archivo."""
    ck = torch.load(ruta, map_location=device, weights_only=False)
    pesos = ck["G"] if isinstance(ck, dict) and "G" in ck else ck
    G = GeneradorUNet(1, 1, ngf).to(device)
    G.load_state_dict(pesos)
    G.eval()
    return G

## 1.6 Evaluación y gráficos

ver_muestras muestra las imágenes guardadas durante el entrenamiento. evaluar_conjunto calcula las métricas de cada par. comparar_visual muestra la fase, el holograma real, el generado, su diferencia y los espectros. comparar_fases muestra la fase de entrada junto a la fase recuperada del holograma real y del generado. revisar_modelo resume en seis revisiones si un experimento funciona bien o mal.

In [ ]:
def evaluar_conjunto(G, df, n=None):
    """Métricas por imagen sobre n pares de df, o sobre todos si n es None.
    Devuelve la tabla y las fases, hologramas reales y generados como arreglos."""
    fases, holos = cargar_pares(df, n)
    gens = predecir(G, fases)
    fases_np, holos_np, gens_np = (x.squeeze(1).numpy() for x in (fases, holos, gens))

    filas = []
    for fase, real, gen in zip(fases_np, holos_np, gens_np):
        mse = float(((gen - real) ** 2).mean())
        pr, pg = metricas_picos(real), metricas_picos(gen)
        fr = comparar_fase(reconstruir_fase(real), fase)
        fg = comparar_fase(reconstruir_fase(gen), fase)
        filas.append(dict(
            mae=float(np.abs(gen - real).mean()),
            psnr=10 * np.log10(4.0 / mse) if mse > 0 else np.inf,
            prom_real=pr["prominencia"], prom_gen=pg["prominencia"],
            dist_real=pr["dist"], dist_gen=pg["dist"],
            r2_real=fr["r2"], nrmse_real=fr["nrmse"],
            r2_gen=fg["r2"], nrmse_gen=fg["nrmse"],
        ))
    return pd.DataFrame(filas), fases_np, holos_np, gens_np


def resumen_evaluacion(tabla):
    """Mediana y cuartiles (Q1, Q3) de cada métrica."""
    return tabla.quantile([0.5, 0.25, 0.75]).T.set_axis(["mediana", "q1", "q3"], axis=1).round(3)


def ver_muestras(exp, epocas=None):
    """Muestras guardadas en la carpeta muestras (fase / generado / real).
    Por defecto muestra un tercio, dos tercios y el final de las épocas entrenadas: con 15, la 5, la 10 y la 15."""
    carpeta = exp["carpeta"] / "muestras"
    guardadas = sorted(int(p.stem.split("_")[1]) for p in carpeta.glob("epoca_*.png"))
    if not guardadas:
        print("No hay muestras en", carpeta, "- entrena primero con entrenar(...).")
        return
    if epocas is None:
        ultima = guardadas[-1]
        epocas = sorted({max(1, round(ultima / 3)), max(1, round(2 * ultima / 3)), ultima})
    fig, ejes = plt.subplots(len(epocas), 1, figsize=(10, 7.5 * len(epocas)))
    ejes = np.atleast_1d(ejes)
    for eje, e in zip(ejes, epocas):
        eje.imshow(Image.open(carpeta / f"epoca_{e:03d}.png").convert("L"), cmap="gray")
        eje.set_title(f"Época {e} (arriba: fase | medio: generado | abajo: real)")
        eje.axis("off")
    plt.tight_layout()
    plt.show()


def graficar_curvas(hist, titulo=""):
    """Curvas de aprendizaje, pérdidas adversarias y métricas de lo generado."""
    ep = np.arange(1, len(hist["loss_D"]) + 1)
    fig, ejes = plt.subplots(2, 3, figsize=(16, 8))

    ejes[0, 0].plot(ep, hist["loss_L1"], label="L1 entrenamiento")
    ejes[0, 0].plot(ep, hist["val_L1"], label="L1 validación")
    ejes[0, 0].set_title("Curva de aprendizaje, L1"); ejes[0, 0].legend()

    ejes[0, 1].plot(ep, hist["loss_D"], label="Discriminador")
    ejes[0, 1].plot(ep, hist["loss_G_gan"], label="Generador, término adversario")
    ejes[0, 1].set_title("Pérdidas adversarias"); ejes[0, 1].legend()

    ejes[0, 2].plot(ep, hist["D_real"], label="D(real)")
    ejes[0, 2].plot(ep, hist["D_falsa"], label="D(falsa)")
    ejes[0, 2].axhline(0.5, color="gray", ls=":")
    ejes[0, 2].set_title("Probabilidad media asignada por D"); ejes[0, 2].legend()

    ejes[1, 0].plot(ep, hist["val_prominencia"], color="green")
    ejes[1, 0].set_title("Prominencia del pico lateral, validación")

    ejes[1, 1].plot(ep, hist["val_nrmse_fase"], color="purple")
    ejes[1, 1].set_title("Error de fase nrmse, validación")

    ejes[1, 2].plot(ep, hist["loss_fase"], label="entrenamiento")
    ejes[1, 2].plot(ep, hist["val_L_fase"], label="validación")
    ejes[1, 2].set_title("Pérdida de fase"); ejes[1, 2].legend()

    for e in ejes.ravel():
        e.set_xlabel("época")
    fig.suptitle(titulo)
    plt.tight_layout()
    plt.show()

def graficar_metricas(tabla, titulo=""):
    """Boxplots de r2 y nrmse de la fase, y scatter de la prominencia del pico lateral."""
    fig, ejes = plt.subplots(1, 3, figsize=(15, 4.5))

    for eje, met in zip(ejes[:2], ["r2", "nrmse"]):
        datos = [tabla[f"{met}_real"].dropna(), tabla[f"{met}_gen"].dropna()]
        eje.boxplot(datos, labels=["Real (piso)", "Generado"], showmeans=False, showfliers=False)
        eje.set_title(f"{met} de la fase")
        eje.grid(alpha=0.3)

    ejes[2].scatter(tabla["prom_real"], tabla["prom_gen"], s=12, alpha=0.5)
    lim = [0, max(tabla["prom_real"].max(), tabla["prom_gen"].max())]
    ejes[2].plot(lim, lim, "r--", lw=1, label="igual al real")
    ejes[2].set_xlabel("Prominencia, real")
    ejes[2].set_ylabel("Prominencia, generado")
    ejes[2].set_title("Prominencia del pico lateral")
    ejes[2].legend()

    fig.suptitle(titulo)
    plt.tight_layout()
    plt.show()

def comparar_visual(fases, reales, generados, indices):
    """Por cada índice muestra la fase, el holograma real, el generado, la diferencia y los dos espectros."""
    fig, ejes = plt.subplots(len(indices), 6, figsize=(20, 3.4 * len(indices)))
    ejes = np.asarray(ejes).reshape(len(indices), 6)
    for fila, i in zip(ejes, indices):
        fila[0].imshow(fases[i], cmap="gray");     fila[0].set_title("Fase de entrada")
        fila[1].imshow(reales[i], cmap="gray");    fila[1].set_title("Holograma real")
        fila[2].imshow(generados[i], cmap="gray"); fila[2].set_title("Holograma generado")
        fila[3].imshow(np.abs(generados[i] - reales[i]), cmap="magma"); fila[3].set_title("Diferencia")
        fila[4].imshow(espectro_log(reales[i]), cmap="gray");    fila[4].set_title("Fourier real")
        fila[5].imshow(espectro_log(generados[i]), cmap="gray"); fila[5].set_title("Fourier generado")
        for e in fila:
            e.axis("off")
    plt.tight_layout()
    plt.show()


def comparar_fases(fases, reales, generados, indices):
    """Por cada índice muestra la fase de entrada y la fase recuperada del holograma real y del generado."""
    def alinear(rec, ref):
        # Quita desfase e inclinación y deja el signo de la referencia, para verlas lado a lado.
        h, w = ref.shape
        yy, xx = np.mgrid[:h, :w]
        A = np.column_stack([ref.ravel(), np.ones(h * w), xx.ravel() / w, yy.ravel() / h])
        coef, *_ = np.linalg.lstsq(A, rec.ravel(), rcond=None)
        rec = (rec.ravel() - A[:, 1:] @ coef[1:]).reshape(h, w)
        return rec * (1 if coef[0] >= 0 else -1)

    def zscore(x):
        return (x - x.mean()) / (x.std() + 1e-9)

    fig, ejes = plt.subplots(len(indices), 3, figsize=(12, 4 * len(indices)))
    ejes = np.asarray(ejes).reshape(len(indices), 3)
    for fila, i in zip(ejes, indices):
        ref = fases[i]
        rec_r, rec_g = reconstruir_fase(reales[i]), reconstruir_fase(generados[i])
        r2_r, r2_g = comparar_fase(rec_r, ref)["r2"], comparar_fase(rec_g, ref)["r2"]
        paneles = ((zscore(ref), "Fase de entrada"),
                   (zscore(alinear(rec_r, ref)), f"Recuperada del real\nr2 = {r2_r:.2f}"),
                   (zscore(alinear(rec_g, ref)), f"Recuperada del generado\nr2 = {r2_g:.2f}"))
        for eje, (img, titulo) in zip(fila, paneles):
            eje.imshow(img, cmap="gray", vmin=-3, vmax=3)
            eje.set_title(titulo, fontsize=10)
            eje.axis("off")
    plt.tight_layout()
    plt.show()

---
# 2. Datos

## 2.1 Estructura y emparejamiento

Cada nombre de archivo contiene la fecha y hora del holograma original, la posición del recorte y la versión. El dataset trae 192 hologramas originales con 64 recortes cada uno y 3 versiones, en total 36864 pares. Las versiones A y B son rotaciones o volteos de la original. Cada holograma se une con su fase por original, recorte y versión.

In [ ]:
RUTA_HOLO = RUTA_DATASET / "Holograms"
RUTA_FASE = RUTA_DATASET / "Phase"

df_holo = indexar_carpeta(RUTA_HOLO)
df_fase = indexar_carpeta(RUTA_FASE)
print("Hologramas indexados:", len(df_holo), "| Fases indexadas:", len(df_fase))

df = emparejar(df_holo, df_fase)
print("\nPares por versión:\n", df["variante"].value_counts().to_string())
print("Hologramas originales distintos:", df["original"].nunique())
print("Total de pares:", len(df))       # esperado 36864

Que los nombres coincidan no basta. Se muestran cuatro pares al azar para comprobar que cada fase corresponde a su holograma. Cada columna es un par, con la fase arriba y el holograma abajo.

In [ ]:
mostrar_pares(df, indices=np.random.default_rng(0).choice(len(df), 4, replace=False))

## 2.2 Partición por holograma original

El artículo reparte el 80 por ciento de las imágenes para entrenamiento y el 20 por ciento para validación, pero no dice con qué criterio. Las tres versiones de un recorte son la misma imagen rotada o volteada. Si el reparto fuera imagen por imagen, versiones del mismo recorte quedarían en carpetas distintas, y eso es fuga de datos.

La primera celda mide cuántos recortes tienen versiones en las dos carpetas y lo compara con lo esperado de un reparto aleatorio. En nuestra corrida fueron 5837 de 12288, cerca de lo esperado, así que la carpeta de validación no es independiente de la de entrenamiento.

Por eso se reparten los 192 hologramas originales en entrenamiento, validación y prueba, con 70, 15 y 15 por ciento. Todo lo que viene de un mismo original queda en el mismo conjunto. La partición usa semilla fija y se guarda en un archivo.

In [ ]:
mixtos, total = parches_en_ambas_carpetas(df)
print(f"Recortes con versiones en más de una carpeta: {len(mixtos)} de {total} ({100 * len(mixtos) / total:.1f} %)")

p = df["carpeta"].value_counts(normalize=True).min()
esperado = 1 - (1 - p) ** 3 - p ** 3
print(f"Esperado con reparto aleatorio imagen por imagen: {100 * esperado:.1f} %")

In [ ]:
df = partir_por_original(df, proporciones=(0.70, 0.15, 0.15), semilla=42, ruta_csv=RUTA_SPLIT)

o = {p: set(df.loc[df.particion == p, "original"]) for p in ("train", "val", "test")}
print("Originales compartidos entre train y val, train y test, val y test:",
      len(o["train"] & o["val"]), len(o["train"] & o["test"]), len(o["val"] & o["test"]))   # esperado 0 0 0
print("Máximo de conjuntos por recorte:",
      df.groupby(["original", "a", "b"])["particion"].nunique().max())                      # esperado 1
print("Hologramas originales por conjunto:", {p: len(v) for p, v in o.items()})

## 2.3 Solo la versión original

Las versiones A y B son la misma imagen rotada o volteada. Usarlas repetiría imágenes, así que se conserva solo la versión original de cada recorte.

In [ ]:
df = df[df["variante"] == "orig"].copy().reset_index(drop=True)
print("Pares con versión original:", len(df))
print("Pares por conjunto:", df["particion"].value_counts().to_dict())

t = df_holo.groupby(["original", "a", "b", "variante"])["ruta"].first().unstack().dropna(subset=["orig", "A", "B"]).iloc[0]
fig, ejes = plt.subplots(1, 3, figsize=(9, 3))
for eje, col, titulo in zip(ejes, ["orig", "A", "B"], ["Original", "A: volteo vertical", "B: volteo horizontal"]):
    eje.imshow(leer_png(t[col]), cmap="gray"); eje.set_title(titulo); eje.axis("off")
plt.show()

## 2.4 Una sola diagonal

Las franjas de los hologramas tienen una de dos orientaciones diagonales. La fase de entrada no dice cuál, así que el generador no puede predecirla y la L1 tampoco la distingue.

Para quitar esa ambigüedad se mide el ángulo del pico lateral de cada holograma. Los de ángulo menor a 90 grados se voltean horizontalmente, junto con su fase, y quedan con la misma diagonal que los demás. El volteo se hace al cargar la imagen y no crea imágenes nuevas. El ángulo se guarda en un archivo para no recalcularlo.

Al final se comprueba que los hologramas volteados quedan con un ángulo cercano al de los que no se voltearon.

In [ ]:
RUTA_ANGULOS = RUTA_SALIDA / "angulos_holograma.csv"

cache_ang = {}
if RUTA_ANGULOS.exists():
    cache_ang = pd.read_csv(RUTA_ANGULOS).set_index("ruta_holo")["angulo"].to_dict()

for ruta in df["ruta_holo"]:
    if str(ruta) not in cache_ang:
        cache_ang[str(ruta)] = metricas_picos(leer_png(ruta) * 2 - 1)["angulo"]
pd.Series(cache_ang).rename_axis("ruta_holo").rename("angulo").reset_index().to_csv(RUTA_ANGULOS, index=False)

df["angulo"] = [cache_ang[str(r)] for r in df["ruta_holo"]]
df["voltear"] = df["angulo"] < 90

print("Pares con franjas a 90 grados o más:", int((~df["voltear"]).sum()),
      "| a voltear:", int(df["voltear"].sum()))

muestra = df[df["voltear"]].sample(100, random_state=0)
_, holos_m = cargar_pares(muestra)
ang_volteados = [metricas_picos(h)["angulo"] for h in holos_m.squeeze(1).numpy()]
print("Ángulo mediano sin voltear:", round(float(df.loc[~df["voltear"], "angulo"].median()), 1),
      "| de los volteados, ya volteados:", round(float(np.median(ang_volteados)), 1))

---
# 3. Referencia

## 3.1 Espectro y reconstrucción de un holograma real

El círculo cian debe cubrir el pico central y la cruz roja debe caer sobre un pico lateral. Después se reconstruye la fase de ese holograma con la FFT y se compara con su fase del dataset.

In [ ]:
df_ref = df[df["particion"] == "train"]
fases_ref, holos_ref = cargar_pares(df_ref, 200)
fases_ref_np, holos_ref_np = fases_ref.squeeze(1).numpy(), holos_ref.squeeze(1).numpy()

ver_fourier(holos_ref_np[0], "Holograma real")
verificar_radio(holos_ref_np[0])
print(verificar_reconstruccion(holos_ref_np[0], fases_ref_np[0]))

## 3.2 Piso y control

Se calculan r2 y nrmse sobre 200 pares reales de entrenamiento, que forman el piso. Después se repite emparejando cada holograma con la fase de otro recorte, que es el control. Si los pares están bien emparejados y la reconstrucción funciona, el piso debe ser claramente mejor que el control. Los pares volteados están incluidos, así que esto también comprueba que el volteo conserva la correspondencia entre fase y holograma.

In [ ]:
piso = pd.DataFrame([comparar_fase(reconstruir_fase(h), f) for h, f in zip(holos_ref_np, fases_ref_np)])
control = pd.DataFrame([comparar_fase(reconstruir_fase(h), fases_ref_np[(i + 7) % len(fases_ref_np)])
                        for i, h in enumerate(holos_ref_np)])

print("PISO, holograma real con su fase")
print(piso.agg(["mean", "median"]).round(3).T.to_string())
print("\nCONTROL, holograma real con la fase de otro recorte")
print(control.agg(["mean", "median"]).round(3).T.to_string())

## 3.3 Parámetros de la pérdida de fase

La pérdida de la sección 1.4 necesita la posición del pico lateral, el radio del filtro y la escala entre la fase de entrada y la fase reconstruida. Se estiman con los 200 hologramas reales de entrenamiento de la sección anterior.

La comprobación del final debe dar un valor cercano a 0.3 con los hologramas reales y cercano a 0.6 con una fase ajena. Si no es así, hay un problema de escala y no conviene entrenar.

In [ ]:
def pendiente_a(rec, ref):
    """Escala a del ajuste rec ~ a*ref + b + c*x + d*y."""
    h, w = ref.shape
    yy, xx = np.mgrid[:h, :w]
    A = np.column_stack([ref.ravel(), np.ones(h * w), xx.ravel() / w, yy.ravel() / h])
    coef, *_ = np.linalg.lstsq(A, rec.ravel(), rcond=None)
    return coef[0]


POS_PICO = Counter(metricas_picos(h)["_pos"] for h in holos_ref_np).most_common(1)[0][0]
RADIO_F = float(np.median([metricas_picos(h)["dist"] for h in holos_ref_np])) / 3
A_ABS = float(np.median([abs(pendiente_a(reconstruir_fase(h), f)) for h, f in zip(holos_ref_np, fases_ref_np)]))
print("Pico fijo (iy, ix):", POS_PICO, "| radio del filtro:", round(RADIO_F, 2), "| escala |a|:", round(A_ABS, 3))

with torch.no_grad():
    hr, fr = holos_ref[:64].to(device), fases_ref[:64].to(device)
    L_real = perdida_fase_torch(reconstruir_fase_torch(hr), fr).cpu()
    L_ajena = perdida_fase_torch(reconstruir_fase_torch(hr), fr.roll(7, dims=0)).cpu()
print(f"Pérdida de fase con su fase: {L_real.median():.3f} | con una fase ajena: {L_ajena.median():.3f}")

---
# 4. Entrenamiento

## 4.1 Configuración

Todo lo que cambia entre experimentos está en CFG, que se guarda como config.json en la carpeta del experimento. Hay que cambiar el nombre antes de cada experimento nuevo, porque si no se sobrescribe el anterior.

En diagonal, voltear usa todos los pares y voltea los de la otra diagonal, y filtrar usa solo los pares que ya están en una diagonal. En lambda_fase, un valor de 0 desactiva la pérdida de fase y deja el pix2pix original.

In [ ]:
# ---- Nuevo experimento ----

CFG = dict(
    nombre="pix2pix_diag_fase",
    semilla=0,
    batch=8,
    ngf=64,                 # canales base del generador
    ndf=64,                 # canales base del discriminador
    lr_G=2e-4,              # tasa de aprendizaje del generador
    lr_D=2e-4,              # tasa de aprendizaje del discriminador
    beta1=0.5,              # parámetro beta1 del optimizador Adam
    lambda_l1=100.0,        # peso de la pérdida L1
    diagonal="filtrar",     # voltear o filtrar
    n_val_epoca=256,        # pares de validación para las métricas de cada época
    guardar_cada=5,         # cada cuántas épocas se guarda el generador
    lambda_fase=6.0,        # peso de la pérdida de fase, 0 la desactiva
)

In [ ]:
# ---- Cargar un experimento ----
NOMBRE = "pix2pix_diag_fase"        # carpeta dentro de resultados/
CHECKPOINT = "ultimo.pt"            # o "G_epoca_040.pt", etc.

carpeta = RUTA_SALIDA / NOMBRE

# configuración usada en ese experimento
CFG = json.loads((carpeta / "config.json").read_text())

# parámetros de la pérdida de fase
if "pos_pico" in CFG:
    POS_PICO = tuple(CFG["pos_pico"]); RADIO_F = CFG["radio_filtro"]; A_ABS = CFG["a_abs"]

# generador
ck = torch.load(carpeta / "checkpoints" / CHECKPOINT, map_location=device, weights_only=False)
pesos = ck["G"] if isinstance(ck, dict) and "G" in ck else ck      # ultimo.pt trae dict, G_epoca_XXX.pt solo los pesos
G = GeneradorUNet(1, 1, CFG["ngf"]).to(device)
G.load_state_dict(pesos)
G.eval()

# historial para las curvas
hist = pd.read_csv(carpeta / "historial.csv", index_col="epoca").to_dict("list")

## 4.2 Datos del experimento

Se eligen los pares según CFG y se crea el cargador de entrenamiento. Los valores calculados en la sección 3 se agregan a CFG para que queden guardados con el experimento.

In [ ]:
assert CFG["diagonal"] in ("voltear", "filtrar"), "diagonal debe ser voltear o filtrar"
d = df if CFG["diagonal"] == "voltear" else df[~df["voltear"]]

df_train = d[d["particion"] == "train"]
df_val = d[d["particion"] == "val"]
df_test = d[d["particion"] == "test"]
loader_train = crear_loader(df_train, CFG, entrenamiento=True)

CFG.update(pos_pico=[int(POS_PICO[0]), int(POS_PICO[1])], radio_filtro=RADIO_F, a_abs=A_ABS)

print("Pares -> train:", len(df_train), "| val:", len(df_val), "| test:", len(df_test))
fase, holo = next(iter(loader_train))
print("Lote de fases     :", tuple(fase.shape), "| rango:", round(fase.min().item(), 3), round(fase.max().item(), 3))
print("Lote de hologramas:", tuple(holo.shape), "| rango:", round(holo.min().item(), 3), round(holo.max().item(), 3))
print(json.dumps(CFG, indent=2))

## 4.3 Prueba rápida

Una época de 100 lotes en un experimento aparte, con el nombre de CFG precedido por prueba. Sirve para comprobar que no hay errores y para medir el tiempo. Multiplicando el tiempo de 100 lotes por el número de lotes de una época completa se estima lo que tarda una época.

In [ ]:
exp_prueba = crear_experimento({**CFG, "nombre": "prueba_" + CFG["nombre"]}, df_val)
entrenar(exp_prueba, loader_train, epocas=1, max_lotes=100)
print("Lotes por época completa:", len(loader_train))

## 4.4 Entrenamiento completo

Crea el generador, el discriminador y los optimizadores según CFG, y entrena. La función entrenar continúa desde la última época registrada. Para retomar un entrenamiento interrumpido, se descomenta la línea de cargar_checkpoint.

Las muestras de cada época quedan en la carpeta muestras. El generador final es el de la última época. No se elige por la L1 de validación, porque la L1 no refleja qué tan bien se recupera la fase.

In [ ]:
# crear el experimento
exp = crear_experimento(CFG, df_val)
print("Parámetros del generador:", sum(p.numel() for p in exp["G"].parameters()))
print("Parámetros del discriminador:", sum(p.numel() for p in exp["D"].parameters()))
print("Salida en:", exp["carpeta"])

In [ ]:
# cargar el último checkpoint. Ejecutar SOLO si se quiere continuar un entrenamiento previo
cargar_checkpoint(exp)

In [ ]:
# entrenar el generador y discriminador
entrenar(exp, loader_train, epocas=20)

---
# 5. Resultados

## 5.1 Curvas de entrenamiento

Las pérdidas de una GAN oscilan y por sí solas no dicen si el modelo mejora. La curva de L1 de entrenamiento contra validación muestra el sobreajuste. La prominencia del pico, el error de fase y la pérdida de fase miden la propiedad física de lo generado.

In [ ]:
graficar_curvas(exp["hist"], titulo=CFG["nombre"])

## 5.2 Evolución de las muestras

En la carpeta muestras hay una imagen por época. Aquí se muestran un tercio, dos tercios y el final del entrenamiento: con 15 épocas, la 5, la 10 y la 15. Cada imagen tiene las fases de entrada (arriba), los hologramas generados (medio) y los reales (abajo), con los mismos cuatro pares. Para ver otras épocas, ver_muestras(exp, [1, 7, 12]).

In [ ]:
ver_muestras(exp)

## 5.3 Generador a evaluar

In [ ]:
G = exp["G"]        # El generador que se acaba de entrenar
# G = cargar_generador(RUTA_SALIDA / "pix2pix_diag_fase_l6" / "checkpoints" / "ultimo.pt", ngf=64)      # Cargar un generador anterior

## 5.4 Evaluación en validación

Se usan todos los pares de validación. El mae y el psnr miden el parecido píxel a píxel con el real. La prominencia y la distancia comparan el pico lateral, es decir, la intensidad y la frecuencia de las franjas. El r2 y el nrmse de la fase del generado se leen frente a los del holograma real, que son el piso.

In [ ]:
tabla_val, fases_v, reales_v, gens_v = evaluar_conjunto(G, df_val)
print("Pares evaluados:", len(tabla_val))
print(resumen_evaluacion(tabla_val).to_string())
graficar_metricas(tabla_val, titulo="Validación")

## 5.5 Evaluación en prueba

El conjunto de prueba se usa una sola vez, con el generador ya elegido. Esta sección solo calcula las métricas y guarda el resumen. Las gráficas y la interpretación están en la sección 6.

In [ ]:
tabla_test, fases_t, reales_t, gens_t = evaluar_conjunto(G, df_test)
resumen_test = resumen_evaluacion(tabla_test)
(RUTA_SALIDA / CFG["nombre"]).mkdir(parents=True, exist_ok=True)
resumen_test.to_csv(RUTA_SALIDA / CFG["nombre"] / "evaluacion_test.csv")
print("Pares evaluados:", len(tabla_test))
print(resumen_test.to_string())
graficar_metricas(tabla_test, titulo="Test")

## 5.6 Comparación visual

Cuatro pares de prueba repartidos de forma uniforme, sin seleccionar los más favorables. La primera figura muestra los hologramas y sus espectros, y la segunda las fases recuperadas.

In [ ]:
indices = np.linspace(0, len(fases_t) - 1, 4).astype(int)
comparar_visual(fases_t, reales_t, gens_t, indices)

In [ ]:
comparar_fases(fases_t, reales_t, gens_t, indices)